# Lesson 7 — Convexity

## Learning objectives

1. State the definitions of convex set and convex function.
2. List operations that preserve convexity.
3. Recognize convexity in a problem written down on paper.
4. Understand why convexity → "any local optimum is global" and what that buys you computationally.

## 1. Definitions

A set $C \subseteq \mathbb{R}^n$ is **convex** if for every $x, y \in C$ and $\theta \in [0, 1]$, $\theta x + (1 - \theta) y \in C$.

A function $f : \mathbb{R}^n \to \mathbb{R}$ is **convex** if for every $x, y$ and $\theta \in [0, 1]$:

$$f(\theta x + (1 - \theta) y) \le \theta f(x) + (1 - \theta) f(y).$$

Strictly convex if the inequality is strict for $x \ne y, \theta \in (0, 1)$.

**First- and second-order conditions** (for differentiable $f$):

- $f$ convex iff $f(y) \ge f(x) + \nabla f(x)^\top (y - x)$ everywhere (tangent below).
- $f \in C^2$ convex iff $\nabla^2 f(x) \succeq 0$ (Hessian PSD) everywhere.

See {cite:t}`Boyd2004` Ch. 3 and {cite:t}`Rockafellar1970`.

## 2. Convex problem

A *convex optimization problem* is

$$\min_x f_0(x) \;\;\text{s.t.}\;\; f_i(x) \le 0, \; A x = b$$

with $f_0, f_i$ convex. The feasible set $\{x : f_i(x) \le 0, Ax = b\}$ is then convex.

**The big payoff:** any local minimum is global. There are no "false summits" {cite:p}`Boyd2004`.

Algorithmically: gradient descent, Newton's method, interior-point methods all converge to the global optimum from any starting point. Polynomial-time complexity (in fixed precision).

## 3. Operations that preserve convexity

If $f, g$ convex, then so are:

- $\alpha f$ for $\alpha \ge 0$
- $f + g$
- $\max(f, g)$
- $f \circ A$ (composition with affine map)
- $\sup_y f(x, y)$ (pointwise supremum)
- $-\log f$ if $f$ is *log-concave*

This **calculus of convexity** lets you check convexity of a complex expression by inspection. It's also how *disciplined convex programming* (CVX, CVXPY) works.

## 4. Pop quiz

| Function | Convex? |
|----------|---------|
| $x^2$ | yes |
| $|x|$ | yes |
| $-\log x$ on $x > 0$ | yes |
| $\log(1 + e^x)$ | yes (softplus) |
| $\frac{1}{x}$ on $x > 0$ | yes |
| $\sqrt{x}$ on $x \ge 0$ | no — *concave* |
| $x \log x$ on $x > 0$ | yes |
| $\sin x$ | no |

`discopt` runs a convexity classifier internally during `m.solve()` — a convex model takes a *fast path* that solves the NLP directly without B&B. Check `r.convex_fast_path` after solving to confirm. Trust your eye too: a sum of *one* nonconvex term ruins everything {cite:p}`Maranas1997`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model, log, exp

m = Model("convex_demo")
x = m.continuous("x", lb=0.01, ub=10)
y = m.continuous("y", lb=-5, ub=5)
# This is convex (softplus is convex, (x-2)**2 is convex, sum-of-convex is convex)
m.minimize(log(1 + exp(y)) + (x - 2)**2)
r = m.solve()
print("status:", r.status, " convex fast path used:", r.convex_fast_path)


## 5. Quasi-convex, log-convex, and other generalizations

Several weaker notions are useful:

- **Quasi-convex:** sublevel sets are convex. Single-mode functions; bisection-friendly. {cite:p}`Boyd2004` Ch 3.
- **Log-convex:** $\log f$ is convex. Common in statistics; products are log-convex.
- **DC (difference of convex):** $f = g - h$ with $g, h$ convex. *Many* nonconvex problems decompose this way; gives algorithmic handles {cite:p}`HorstTuy1996`.

## References
{cite:p}`Boyd2004` is the standard text. {cite:p}`Rockafellar1970` for the deep theory.